In [1]:
import xarray as xr
import cftime

In [2]:
ap6_folder_future = "/hkfs/work/workspace/scratch/ou4895-for_lukas/SSP585/ap6"
#ap8_folder_future = "/hkfs/work/workspace/scratch/ou4895-for_lukas/SSP585/ap8"
apk_folder_future = "/hkfs/work/workspace/scratch/ou4895-for_lukas/SSP585/apk"
apc_folder_future = "/hkfs/work/workspace/scratch/ou4895-for_lukas/SSP585/apc"
ap6_folder_hist = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/ap6"
# ap8_folder_hist = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/ap8"
ap8_folder_hist = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/ap8_supplement"
ap8_folder_future = "/hkfs/work/workspace/scratch/ou4895-for_lukas/SSP585/ap8_supplement"
apk_folder_hist = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/apk"
apc_folder_hist = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/apc"
out = '/hkfs/work/workspace/scratch/xo8179-ukesm/not_regridded/'

In [3]:
def load_x_files_from_folder(folder:str, first=0, last=-1, debug=False):
    import os
    import glob

    # Get all .nc files in the folder
    files = sorted(glob.glob(os.path.join(folder, "*.nc")))
    print(f'Found {len(files)} in specified folder.')

    # Select the last 5 files
    if first < 0 or last < 0:
        if first < 0 and last < 0:
            print(f'Attempting to load {last - first} files...')
            selected_files = files[first:last]
        else:
            print(f'Attempting to load {len(files) - first} files...')
            selected_files = files[first:]
    else:
        print(f'Attempting to load {last - first} files...')
        selected_files = files[first:last]
    if debug:
        print(selected_files)
    # Load them as a combined dataset
    ds = xr.open_mfdataset(selected_files,
                           parallel=True, # Enable parallel reads
                           chunks='auto', # Let dask decide chunking
                           combine='by_coords')

    return ds

# Loading apc

In [ ]:
import os
import glob
import xarray as xr

def save_subset(ds, vars_to_select, i, append_dim, name):
    ds_selected = None
    rename_dims = {}
    try:
        ds_selected = ds[vars_to_select]
    except Exception:
        return False
    if ds_selected:
        if i == 0:
            ds_selected.to_zarr(output_path+name+'.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+name+'.zarr', mode="a", append_dim=append_dim)
    return True

# Input folder
folder = apc_folder_future
# Output path
output_path = '/hkfs/work/workspace/scratch/xo8179-ukesm/not_regridded/SSP585/apc/'

# Get all NetCDF files, sorted
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print(f"Found {len(files)} files")

# Loop through files and append to Zarr store
for i, f in enumerate(files):
    if i < 151:
        continue
    print(f"[{i+1}/{len(files)}] Processing")

    ds = xr.open_dataset(f, chunks={})   # open one file only
    expected_vars = ['wind','ht_1','p_1','u','v','dz_dt','temp','q','psag','rvor','rvor_1','rvor_2','rvor_3']
    unexpected_vars=[]
    for var in ds.data_vars:
        if not var in expected_vars:
            unexpected_vars.append(var)
    
    not_found = []
    filename = f.split('/')[-1]
    
    if not save_subset(ds, ['wind'], i, 't', 'wind'):
        not_found.append('wind')

    if not save_subset(ds, ['u', 'v'], i, 't_1', 'uv'):
        not_found.append('uv')
    
    if not save_subset(ds, ['rvor', 'rvor_1', 'rvor_2', 'rvor_3'], i, 't_1', 'rvor'):
        not_found.append('rvor')

    if not save_subset(ds, ['q'], i, 't_1', 'q'):
        not_found.append('q')

    if not save_subset(ds, ['temp'], i, 't_1', 'temp'):
        not_found.append('temp')

    if not save_subset(ds, ['psag'], i, 't_1', 'psag'):
        not_found.append('psag')

    if not save_subset(ds, ['dz_dt'], i, 't_1', 'dz_dt'):
        not_found.append('dz_dt')

    if not save_subset(ds, ['p_1'], i, 't_1', 'p_1'):
        not_found.append('p_1')

    if not save_subset(ds, ['ht_1'], i, 't_1', 'ht_1'):
        not_found.append('ht_1')
        
    if len(not_found) > 0:
        print(f'Did not find {str(not_found)} in {filename}')
    if len(unexpected_vars) > 0:
        print(f'Found unexpected vars {str(unexpected_vars)} in {filename}')
    ds.close()

print("✅ All files merged into Zarr:", output_path)

# Loading apk

In [ ]:
import os
import glob
import xarray as xr

# Input folder
folder = apk_folder_future
# Output path
output_path = '/hkfs/work/workspace/scratch/xo8179-ukesm/not_regridded/SSP585/apk/'
start = 800

# Get all NetCDF files, sorted
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print(f"Found {len(files)} files")

# Loop through files and append to Zarr store
for i, f in enumerate(files):
    if i<start:
        continue
    print(f"[{i+1}/{len(files)}] Processing")

    ds = xr.open_dataset(f, chunks={})   # open one file only
    expected_vars = ['u', 'v', 'precip']
    unexpected_vars=[]
    for var in ds.data_vars:
        if not var in expected_vars:
            unexpected_vars.append(var)
    
    not_found = []
    filename = f.split('/')[-1]
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['precip']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('precip')
    if ds_selected:
        if "longitude" in ds_selected.dims:
            rename_dims["longitude"] = "longitude_1"
            ds_selected = ds_selected.rename(rename_dims)
        if i == start:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="a", append_dim="t")
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['u', 'v']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('uv')
    if ds_selected:
        if i == start:
            ds_selected.to_zarr(output_path+'uv.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'uv.zarr', mode="a", append_dim="t")
    
        
    if len(not_found) > 0:
        print(f'Did not find {str(not_found)} in {filename}')
    if len(unexpected_vars) > 0:
        print(f'Found unexpected vars {str(unexpected_vars)} in {filename}')
    ds.close()

print("✅ All files merged into Zarr:", output_path)

# Loading ap8

In [ ]:
import os
import glob
import xarray as xr

# Input folder
folder = ap8_folder_future
# Output path
output_path = '/hkfs/work/workspace/scratch/xo8179-ukesm/not_regridded/SSP585/ap8/'

# Get all NetCDF files, sorted
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print(f"Found {len(files)} files")
start=120

# Loop through files and append to Zarr store
for i, f in enumerate(files):
    if i<start:
        continue
    filename = f.split('/')[-1]
    
    try:

        ds = xr.open_dataset(f, chunks={})   # open one file only
    except Exception:
        print(f"[{i+1}/{len(files)}] EXCEPTION, cannot open {filename}")
        continue
    print(f"[{i+1}/{len(files)}] Processing {filename}")
        
    expected_vars = ['temp', 'precip']
    unexpected_vars=[]
    for var in ds.data_vars:
        if not var in expected_vars:
            unexpected_vars.append(var)
    
    not_found = []

    SAVE = True
    
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['precip']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('precip')
    #if ds_selected:
    if SAVE:
        if i == start:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="a", append_dim="t")
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['temp']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('temp')
    # if ds_selected:
    if SAVE:
        if i == start:
            ds_selected.to_zarr(output_path+'temp.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'temp.zarr', mode="a", append_dim="t_1")
    
        
    if len(not_found) > 0:
        print(f'Did not find {str(not_found)} in {filename}')
    #if len(unexpected_vars) > 0:
        #print(f'Found unexpected vars {str(unexpected_vars)} in {filename}')
    ds.close()

print("✅ All files merged into Zarr:", output_path)

# Loading ap6

In [ ]:
import os
import glob
import xarray as xr

# Input folder
folder = ap6_folder_future
# Output path
output_path = '/hkfs/work/workspace/scratch/xo8179-ukesm/not_regridded/SSP585/ap6/'

# Get all NetCDF files, sorted
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print(f"Found {len(files)} files")

# Loop through files and append to Zarr store
for i, f in enumerate(files):
    if i<623:
        continue
    print(f"[{i+1}/{len(files)}] Processing")

    ds = xr.open_dataset(f, chunks={})   # open one file only
    expected_vars = ['temp', 'temp_1', 'temp_2', 'precip', 'u', 'v','ht', 'ht_1', 'ht_2', 'ht_3', 'ht_4', 'ht_5']
    unexpected_vars=[]
    for var in ds.data_vars:
        if not var in expected_vars:
            unexpected_vars.append(var)
    
    not_found = []
    filename = f.split('/')[-1]
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['temp', 'temp_1', 'temp_2']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('temps')
    if ds_selected:
        if "longitude" in ds_selected.dims:
            rename_dims["longitude"] = "longitude_1"
            ds_selected = ds_selected.rename(rename_dims)
        if i == 0:
            ds_selected.to_zarr(output_path+'temps.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'temps.zarr', mode="a", append_dim="t")
    
    ds_selected = None
    rename_dims = {}
    try:
        vars = ['precip']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('precip')
    if ds_selected:
        if "longitude" in ds_selected.dims:
            rename_dims["longitude"] = "longitude_1"
            ds_selected = ds_selected.rename(rename_dims)
        if i == 0:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'precip.zarr', mode="a", append_dim="t")
    

    ds_selected = None
    rename_dims = {}
    try:
        vars = ['u', 'v']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('uv')
    if ds_selected:
        if i == 0:
            ds_selected.to_zarr(output_path+'uv.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'uv.zarr', mode="a", append_dim="t")
    
        
    if "ht" in ds.data_vars:
        rename_vars = {
            "ht": "ht_1",
            "ht_1": "ht_2",
            "ht_2": "ht_3",
            "ht_3": "ht_4",
            "ht_4": "ht_5",
        }
        ds = ds.rename(rename_vars)

        rename_dims = {}
        if "longitude" in ds.dims:
            rename_dims["longitude"] = "longitude_1"
        if "longitude_1" in ds.dims:
            rename_dims["longitude_1"] = "longitude_2"
        if rename_dims:
            ds = ds.rename(rename_dims)

    ds_selected = None
    try:
        vars = ['ht_1', 'ht_2', 'ht_3', 'ht_4', 'ht_5']
        ds_selected = ds[vars]
    except Exception:
        not_found.append('hts')
    if ds_selected:
        if i == 0:
            ds_selected.to_zarr(output_path+'hts.zarr', mode="w")
        else:
            ds_selected.to_zarr(output_path+'hts.zarr', mode="a", append_dim="t")
    

    if len(not_found) > 0:
        print(f'Did not find {str(not_found)} in {filename}')
    if len(unexpected_vars) > 0:
        print(f'Found unexpected vars {str(unexpected_vars)} in {filename}')
    ds.close()

print("✅ All files merged into Zarr:", output_path)

# Missing values

## apk
historical:
- No u and v in 1984

future:
- No u and v in 2099

## ap6
historical:
- From 1980: Only ht variables (except 1984)
- In 1984: No ht variables (only temp and precip)
- 1991 is fine

# Done
- ap8 (hist, future)
- apk (hist, future)
- ap6 (hist, future)
- apc (hist, future)


# Select the useful data

# Height

### Hist

In [31]:
levels = [925, 850, 700, 500, 300, 200]
future_start=cftime.Datetime360Day(2095, 1, 1, 0)
future_end=cftime.Datetime360Day(2101, 1, 1, 0)

In [ ]:

ds = xr.open_dataset(out+'Hist/ap6/hts.zarr')
print(ds)

In [ ]:
ds_selected = ds['ht_3']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
ds_selected.to_zarr(out + 'final/pl_height.zarr')

### Future

In [31]:
ds = xr.open_dataset(out+'SSP585/ap6/hts.zarr')
print(ds)

<xarray.Dataset> Size: 106GB
Dimensions:      (t: 30960, p: 3, latitude: 144, longitude_1: 192, p_1: 2,
                  p_2: 18, p_3: 19, longitude_2: 1, p_4: 8)
Coordinates:
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * longitude_2  (longitude_2) float32 4B 357.2
  * p            (p) float32 12B 10.0 100.0 1e+03
  * p_1          (p_1) float32 8B 500.0 1e+03
  * p_2          (p_2) float32 72B 925.0 850.0 700.0 600.0 ... 20.0 10.0 5.0 1.0
  * p_3          (p_3) float32 76B 1e+03 925.0 850.0 700.0 ... 20.0 10.0 5.0 1.0
  * p_4          (p_4) float32 32B 1e+03 850.0 700.0 500.0 250.0 100.0 50.0 10.0
  * t            (t) object 248kB 2015-01-01 12:00:00 ... 2100-12-30 12:00:00
Data variables:
    ht_1         (t, p, latitude, longitude_1) float32 10GB ...
    ht_2         (t, p_1, latitude, longitude_1) float32 7GB ...
    ht_3         (t, p_2, latitude, longitude_1) float32

In [ ]:
ds_selected = ds['ht_3']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
ds_selected.to_zarr(out + 'final/pl_height_future.zarr', mode='w')

In [ ]:
ds = xr.open_dataset(out + 'final/pl_height.zarr')
print(ds)

<xarray.Dataset> Size: 86GB
Dimensions:      (t: 43200, p_2: 18, latitude: 144, longitude_1: 192)
Coordinates:
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * p_2          (p_2) float32 72B 925.0 850.0 700.0 600.0 ... 20.0 10.0 5.0 1.0
  * t            (t) object 346kB 1979-01-01 12:00:00 ... 2100-12-30 12:00:00
Data variables:
    ht_3         (t, p_2, latitude, longitude_1) float32 86GB ...


# U an V

## 3D var

### Hist

In [38]:
ds = xr.open_dataset(out+'Hist/apc/uv.zarr')
print(ds)

<xarray.Dataset> Size: 69GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 51840)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    u          (t_1, p_2, latitude, longitude) float32 35GB ...
    v          (t_1, p_2, latitude, longitude) float32 35GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [10]:
ds_selected = ds['u']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/u.zarr', mode='w')

In [11]:
ds_selected = ds['v']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/v.zarr', mode='w')

### Future

In [39]:
ds = xr.open_dataset(out+'SSP585/apc/uv.zarr')
print(ds)

<xarray.Dataset> Size: 166GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 124200)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    u          (t_1, p_2, latitude, longitude) float32 83GB ...
    v          (t_1, p_2, latitude, longitude) float32 83GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [40]:
ds_selected = ds['u']
ds_selected = ds_selected.sel(t_1=slice(future_start,future_end))
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
# print(ds_selected)
ds_selected.to_zarr(out + 'final/u_future.zarr', mode='w')

<xarray.DataArray 'u' (t: 8641, p_2: 6, latitude: 145, longitude: 192)> Size: 6GB
[1443392640 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t          (t) object 69kB 2095-01-01 00:00:00 ... 2101-01-01 00:00:00
Attributes:
    date:           01/01/15
    long_name:      Westerly component of wind  u
    name:           u
    source:         Unified Model Output (Vn11.2):
    standard_name:  eastward_wind
    time:           00:00
    title:          Westerly component of wind  u
    units:          m s-1
    valid_max:      107.173583984375
    valid_min:      -55.618408203125


In [41]:
ds_selected = ds['v']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
ds_selected = ds_selected.sel(t_1=slice(future_start,future_end))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/v_future.zarr', mode='w')

## Surface

In [36]:
import pandas as pd

def calc_avg_wind(ds_test):

    ds_test = ds_test.sortby("t")
    
    # --- Shift by one step so each timestep aligns with its successor ---
    ds_shifted = ds_test.shift(t=+1)
    
    # --- Average pairs of consecutive 3-hourly values ---
    ds_avg = (ds_test + ds_shifted) / 2.0
    
    # --- Keep only every second timestep (the “next available” one) ---
    ds_avg = ds_avg.isel(t=slice(1, None, 2))
    
    # --- Create new timestamps (06:00, 12:00, 18:00, 00:00 next day) ---
    new_times = []
    for t in ds_avg["t"].values:
        # t is a cftime.Datetime360Day object
        # Add 1.5 hours to go from 04:30 → 06:00, etc.
        new_hour = (t.hour + 2) % 24
        new_day = t.day if t.hour < new_hour else t.day+1
        # Wrap around month/day if needed
        if new_day > 30:  # 360-day calendar → 30 days/month
            new_day -= 30
            new_month = t.month + 1
            if new_month > 12:
                new_month = 1
                new_year = t.year + 1
            else:
                new_year = t.year
        else:
            new_month = t.month
            new_year = t.year
        new_times.append(
            cftime.Datetime360Day(new_year, new_month, new_day, new_hour, 0, 0)
        )
    
    # --- Assign corrected time coordinates ---
    ds_avg = ds_avg.assign_coords(t=("t", new_times))
    return ds_avg

### Hist

In [34]:
ds = xr.open_dataset(out+'Hist/apk/uv.zarr')
new_ds = ds.sortby('t')
print(new_ds)

<xarray.Dataset> Size: 22GB
Dimensions:      (ht: 1, latitude: 144, latitude_1: 145, longitude: 192,
                  longitude_1: 192, t: 100800)
Coordinates:
  * ht           (ht) float32 4B -1.0
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * latitude_1   (latitude_1) float32 580B -90.0 -88.75 -87.5 ... 88.75 90.0
  * longitude    (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * t            (t) object 806kB 1979-01-01 01:30:00 ... 2014-12-30 22:30:00
Data variables:
    u            (t, ht, latitude, longitude) float32 11GB ...
    v            (t, ht, latitude_1, longitude_1) float32 11GB ...
Attributes:
    history:  Sun Jul 27 11:44:47 BST 2025 - CONVSH V1.94 03-May-2018


In [37]:
ds_avg = calc_avg_wind(ds_test)
print(ds_avg)

<xarray.Dataset> Size: 2GB
Dimensions:      (ht: 1, latitude: 144, latitude_1: 145, longitude: 192,
                  longitude_1: 192, t: 10080)
Coordinates:
  * ht           (ht) float32 4B -1.0
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * latitude_1   (latitude_1) float32 580B -90.0 -88.75 -87.5 ... 88.75 90.0
  * longitude    (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * t            (t) object 81kB 2094-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    u            (t, ht, latitude, longitude) float32 1GB 2.63 2.838 ... -6.377
    v            (t, ht, latitude_1, longitude_1) float32 1GB -4.304 ... 7.002


In [ ]:
ds_selected = ds_avg['u']
ds_selected = ds_selected.squeeze('ht', drop=True)
print(ds_selected)

In [ ]:
ds_selected.to_zarr(out + 'final/u_surface.zarr', mode='w')

In [ ]:
ds_selected = ds_avg['v']
ds_selected = ds_selected.squeeze('ht', drop=True)
print(ds_selected)

In [ ]:
ds_selected.to_zarr(out + 'final/v_surface.zarr', mode='w')

### Future

In [ ]:
ds = xr.open_dataset(out+'SSP585/apk/uv.zarr')
new_ds = ds.sortby('t')
# print(new_ds)

In [ ]:
# Take a bit more, to account for pieces lost due to calculating the average
slice_start = cftime.Datetime360Day(2094, 1, 1)
slice_end = cftime.Datetime360Day(2102, 1, 1)
ds_test = new_ds.sel(t=slice(slice_start,slice_end))
# print(ds_test)

In [ ]:
ds_avg = calc_avg_wind(ds_test)
# print(ds_avg)

In [ ]:
ds_selected = ds_avg['u']
ds_selected = ds_selected.squeeze('ht', drop=True)
# print(ds_selected)

In [26]:
ds_selected.to_zarr(out + 'final/u_surface_future.zarr', mode='w')

In [ ]:
ds_selected = ds_avg['v']
ds_selected = ds_selected.squeeze('ht', drop=True)
rename_vars = {
    "latitude_1": "latitude",
    "longitude_1": "longitude",
}
ds_selected = ds_selected.rename(rename_vars)
# print(ds_selected)

In [28]:
ds_selected.to_zarr(out + 'final/v_surface_future.zarr', mode='w')

# Temp

## 3D

### Hist

In [4]:
ds = xr.open_dataset(out+'Hist/apc/temp.zarr')
print(ds)

<xarray.Dataset> Size: 35GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 51840)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    temp       (t_1, p_2, latitude, longitude) float32 35GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [5]:
ds_selected = ds['temp']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
ds_selected.to_zarr(out + 'final/temp.zarr', mode='w')

### Future

In [29]:
ds = xr.open_dataset(out+'SSP585/apc/temp.zarr')
print(ds)

<xarray.Dataset> Size: 83GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 124200)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    temp       (t_1, p_2, latitude, longitude) float32 83GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [32]:
ds_selected = ds['temp']
ds_selected.sel(t_1=slice(future_start, future_end))
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
print(ds_selected)
ds_selected.to_zarr(out + 'final/temp_future.zarr', mode='w')

<xarray.DataArray 'temp' (t: 124200, p_2: 6, latitude: 145, longitude: 192)> Size: 83GB
[20746368000 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t          (t) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Attributes:
    date:           01/01/15
    long_name:      Temperature T
    name:           temp
    source:         Unified Model Output (Vn11.2):
    standard_name:  air_temperature
    time:           00:00
    title:          Temperature T
    units:          K
    valid_max:      314.69580078125
    valid_min:      0.0


## Surface

In [ ]:
ds = xr.open_dataset(out+'Hist/ap8/temp.zarr')
# print(ds)
slice_start = cftime.Datetime360Day(1985, 1, 1, 6)
slice_end = cftime.Datetime360Day(2015, 1, 1)
ds_test = ds.sel(t_1=slice(slice_start,slice_end))
# print(ds_test)
ds = ds_test.sel(t_1=ds_test.t_1.where((ds_test.t_1.dt.hour % 6 == 0), drop=True))
# print(ds)
ds_selected = ds['temp']
ds_selected = ds_selected.squeeze('ht', drop=True)

rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
print(ds_selected)

<xarray.Dataset> Size: 11GB
Dimensions:    (ht: 1, latitude: 144, longitude: 192, t_1: 103680)
Coordinates:
  * ht         (ht) float32 4B -1.0
  * latitude   (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude  (longitude) float32 768B 0.9375 2.812 4.688 ... 355.3 357.2 359.1
  * t_1        (t_1) object 829kB 1979-01-01 03:00:00 ... 2015-01-01 00:00:00
Data variables:
    temp       (t_1, ht, latitude, longitude) float32 11GB ...
Attributes:
    history:  Sat Oct 18 12:02:20 BST 2025 - CONVSH V1.94 03-May-2018


In [17]:
ds_selected.to_zarr(out + 'final/temp_surface.zarr', mode='w')

### Future

In [42]:
ds = xr.open_dataset(out+'SSP585/ap8/temp.zarr')
# print(ds)
ds_test = ds.sel(t_1=slice(future_start,future_end))
# print(ds_test)
ds = ds_test.sel(t_1=ds_test.t_1.where((ds_test.t_1.dt.hour % 6 == 0), drop=True))
# print(ds)
ds_selected = ds['temp']
ds_selected = ds_selected.squeeze('ht', drop=True)

rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
print(ds_selected)

<xarray.DataArray 'temp' (t: 8641, latitude: 144, longitude: 192)> Size: 956MB
[238906368 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude  (longitude) float32 768B 0.9375 2.812 4.688 ... 355.3 357.2 359.1
  * t          (t) object 69kB 2095-01-01 00:00:00 ... 2101-01-01 00:00:00
Attributes:
    date:           01/01/15
    long_name:      Temperature T
    name:           temp
    source:         Unified Model Output (Vn11.2):
    standard_name:  air_temperature
    time:           00:00
    title:          Temperature T
    units:          K
    valid_max:      326.078125
    valid_min:      195.00439453125


In [43]:
ds_selected.to_zarr(out + 'final/temp_surface_future.zarr', mode='w')

# Precip Surface

In [66]:
import xarray as xr
import cftime

def calculate_accum_precip(ds):
    # --- Sort by time ---
    ds = ds.sortby("t")
    
    # --- Shift by one timestep to align pairs (e.g., 01:30 with 04:30) ---
    ds_shifted = ds.shift(t=+1)
    
    # --- Define constants ---
    dt_seconds = 6 * 3600  # 6 hours in seconds
    
    # --- Compute accumulated precipitation over 6 hours ---
    # Trapezoidal integration: mean of two rates × duration
    accum_precip = ((ds["precip"] + ds_shifted["precip"]) / 2.0) * dt_seconds
    
    # --- Keep only every second time step (the “next available” one) ---
    accum_precip = accum_precip.isel(t=slice(1, None, 2))
    
    # --- Create new timestamps for 6-hourly output ---
    new_times = []
    for t in accum_precip["t"].values:
        # Shift timestamps forward to the 6-hour mark (same logic as before)
        new_hour = (t.hour + 2) % 24
        new_day = t.day if t.hour < new_hour else t.day+1
        if new_day > 30:
            new_day -= 30
            new_month = t.month + 1
            if new_month > 12:
                new_month = 1
                new_year = t.year + 1
            else:
                new_year = t.year
        else:
            new_month = t.month
            new_year = t.year
    
        new_times.append(cftime.Datetime360Day(new_year, new_month, new_day, new_hour))
    
    accum_precip = accum_precip.assign_coords(t=("t", new_times))
    
    # --- Create output dataset ---
    ds_accum = xr.Dataset(
        {"precip_accum_6h": accum_precip},
        coords=accum_precip.coords,
        attrs=ds.attrs,
    )
    
    # --- Update metadata ---
    ds_accum["precip_accum_6h"].attrs.update({
        "long_name": "Total precipitation accumulated over 6 hours",
        "units": "kg m-2",
        "description": "Computed from 3-hourly precipitation rate (kg m-2 s-1) using trapezoidal integration."
    })
    return ds_accum

### Hist

In [18]:
ds = xr.open_dataset(out+'Hist/ap8/precip.zarr')
slice_start = cftime.Datetime360Day(1985, 1, 1)
slice_end = cftime.Datetime360Day(2015, 1, 1)
ds_test = ds.sel(t=slice(slice_start,slice_end))
print(ds_test['precip'])

<xarray.DataArray 'precip' (t: 86400, surface: 1, latitude: 144, longitude: 192)> Size: 10GB
[2388787200 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude  (longitude) float32 768B 0.9375 2.812 4.688 ... 355.3 357.2 359.1
  * surface    (surface) float32 4B 0.0
  * t          (t) object 691kB 1985-01-01 01:30:00 ... 2014-12-30 22:30:00
Attributes:
    date:           01/01/50
    long_name:      Total precipitation rate
    name:           precip
    source:         Unified Model Output (Vn10.9):
    standard_name:  precipitation_flux
    time:           00:00
    title:          Total precipitation rate
    units:          kg m-2 s-1
    valid_max:      0.007962364703416824
    valid_min:      0.0


In [53]:
ds_accum = calculate_accum_precip(ds_test)
print(ds_accum)
ds_selected = ds_accum['precip_accum_6h']
ds_selected = ds_selected.squeeze('surface', drop=True)

print(ds_selected)

NameError: name 'ds_test' is not defined

In [23]:
ds_selected.to_zarr(out + 'final/precip_surface.zarr', mode='w')

### Future

In [67]:
ds = xr.open_dataset(out+'SSP585/ap8/precip.zarr')
start = cftime.Datetime360Day(2094, 1, 1)
end = cftime.Datetime360Day(2102, 1, 1)
ds_test = ds.sel(t=slice(start,end))
print(ds_test['precip'])

<xarray.DataArray 'precip' (t: 20160, surface: 1, latitude: 144, longitude: 192)> Size: 2GB
[557383680 values with dtype=float32]
Coordinates:
  * latitude   (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude  (longitude) float32 768B 0.9375 2.812 4.688 ... 355.3 357.2 359.1
  * surface    (surface) float32 4B 0.0
  * t          (t) object 161kB 2094-01-01 01:30:00 ... 2100-12-30 22:30:00
Attributes:
    date:           01/01/15
    long_name:      Total precipitation rate
    name:           precip
    source:         Unified Model Output (Vn11.2):
    standard_name:  precipitation_flux
    time:           00:00
    title:          Total precipitation rate
    units:          kg m-2 s-1
    valid_max:      0.0083646634593606
    valid_min:      0.0


In [68]:
ds_accum = calculate_accum_precip(ds_test)
print(ds_accum)
ds_selected = ds_accum['precip_accum_6h']
ds_selected = ds_selected.squeeze('surface', drop=True)

print(ds_selected)

<xarray.Dataset> Size: 1GB
Dimensions:          (latitude: 144, longitude: 192, surface: 1, t: 10080)
Coordinates:
  * latitude         (latitude) float32 576B -89.38 -88.12 ... 88.12 89.38
  * longitude        (longitude) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * surface          (surface) float32 4B 0.0
  * t                (t) object 81kB 2094-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    precip_accum_6h  (t, surface, latitude, longitude) float32 1GB 0.09733 .....
Attributes:
    history:  Sun Nov  2 16:03:42 GMT 2025 - CONVSH V1.94 03-May-2018
<xarray.DataArray 'precip_accum_6h' (t: 10080, latitude: 144, longitude: 192)> Size: 1GB
array([[[9.73257422e-02, 9.02369991e-02, 9.31771845e-02, ...,
         9.09084976e-02, 1.04779944e-01, 1.09386571e-01],
        [7.98147768e-02, 9.49391425e-02, 1.49407983e-01, ...,
         5.56790158e-02, 9.43479687e-02, 7.98952729e-02],
        [3.69188279e-01, 3.92782897e-01, 4.18875933e-01, ...,
         2.18656987e-01, 3.71455

In [70]:
ds_selected.to_zarr(out + 'final/precip_surface_future.zarr', mode='w')

# Specific Humidity (q)

### Hist

In [9]:
ds = xr.open_dataset(out+'Hist/apc/q.zarr')
print(ds)

<xarray.Dataset> Size: 35GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 51840)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    q          (t_1, p_2, latitude, longitude) float32 35GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [10]:
ds_selected = ds['q']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/q.zarr', mode='w')

### Future

In [50]:
ds = xr.open_dataset(out+'SSP585/apc/q.zarr')
print(ds)

<xarray.Dataset> Size: 83GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, t_1: 124200)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    q          (t_1, p_2, latitude, longitude) float32 83GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [51]:
ds_selected = ds['q']
ds_selected.sel(t_1=slice(future_start, future_end))
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/q_future.zarr', mode='w')

# Relative Vorticity (Rvor)

### Hist

In [13]:
ds = xr.open_dataset(out+'Hist/apc/rvor.zarr')
print(ds)

<xarray.Dataset> Size: 81GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, p_3: 1, t_1: 51840)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * p_3        (p_3) float32 4B -1.0
  * t_1        (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    rvor       (t_1, p_3, latitude, longitude) float32 6GB ...
    rvor_1     (t_1, p_3, latitude, longitude) float32 6GB ...
    rvor_2     (t_1, p_2, latitude, longitude) float32 35GB ...
    rvor_3     (t_1, p_2, latitude, longitude) float32 35GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [14]:
ds_selected = ds['rvor_2']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/rvor.zarr', mode='w')

### Future

In [52]:
ds = xr.open_dataset(out+'SSP585/apc/rvor.zarr')
print(ds)

<xarray.Dataset> Size: 194GB
Dimensions:    (latitude: 145, longitude: 192, p_2: 6, p_3: 1, t_1: 124200)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * p_3        (p_3) float32 4B -1.0
  * t_1        (t_1) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    rvor       (t_1, p_3, latitude, longitude) float32 14GB ...
    rvor_1     (t_1, p_3, latitude, longitude) float32 14GB ...
    rvor_2     (t_1, p_2, latitude, longitude) float32 83GB ...
    rvor_3     (t_1, p_2, latitude, longitude) float32 83GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [56]:
ds_selected = ds['rvor_2']
ds_selected.sel(t_1=slice(future_start, future_end))
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/rvor_future.zarr', mode='w')

# dz_dt

### Hist

In [4]:
ds = xr.open_dataset(out+'Hist/apc/dz_dt.zarr')
print(ds)

<xarray.Dataset> Size: 35GB
Dimensions:    (t_1: 51840, p_2: 6, latitude: 145, longitude: 192)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    dz_dt      (t_1, p_2, latitude, longitude) float32 35GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [5]:
ds_selected = ds['dz_dt']
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/dz_dt.zarr', mode='w')

### Future

In [57]:
ds = xr.open_dataset(out+'SSP585/apc/dz_dt.zarr')
print(ds)

<xarray.Dataset> Size: 83GB
Dimensions:    (t_1: 124200, p_2: 6, latitude: 145, longitude: 192)
Coordinates:
  * latitude   (latitude) float32 580B -90.0 -88.75 -87.5 ... 87.5 88.75 90.0
  * longitude  (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * p_2        (p_2) float32 24B 925.0 850.0 700.0 500.0 300.0 200.0
  * t_1        (t_1) object 994kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    dz_dt      (t_1, p_2, latitude, longitude) float32 83GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [58]:
ds_selected = ds['dz_dt']
ds_selected.sel(t_1=slice(future_start, future_end))
ds_selected = ds_selected.sel(p_2=ds_selected.p_2.isin(levels))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/dz_dt_future.zarr', mode='w')

# Geopotential Height

### Hist

In [17]:
ds = xr.open_dataset(out+'Hist/apc/ht_1.zarr')
print(ds)

<xarray.Dataset> Size: 6GB
Dimensions:      (t_1: 51840, p: 1, latitude_1: 144, longitude_1: 192)
Coordinates:
  * latitude_1   (latitude_1) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * p            (p) float32 4B 500.0
  * t_1          (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    ht_1         (t_1, p, latitude_1, longitude_1) float32 6GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [18]:
ds_selected = ds['ht_1']
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/ht_1.zarr', mode='w')

### Future

In [59]:
ds = xr.open_dataset(out+'SSP585/apc/ht_1.zarr')
print(ds)

<xarray.Dataset> Size: 14GB
Dimensions:      (t_1: 123840, p: 1, latitude_1: 144, longitude_1: 192)
Coordinates:
  * latitude_1   (latitude_1) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * p            (p) float32 4B 500.0
  * t_1          (t_1) object 991kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    ht_1         (t_1, p, latitude_1, longitude_1) float32 14GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [60]:
ds_selected = ds['ht_1']
ds_selected.sel(t_1=slice(future_start, future_end))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/ht_1_future.zarr', mode='w')

# Surface Level Pressure (SLP)

### Hist

In [22]:
ds = xr.open_dataset(out+'Hist/apc/p_1.zarr')
print(ds)

<xarray.Dataset> Size: 6GB
Dimensions:      (latitude_1: 144, longitude_1: 192, msl: 1, t_1: 51840)
Coordinates:
  * latitude_1   (latitude_1) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * msl          (msl) float32 4B 0.0
  * t_1          (t_1) object 415kB 1979-01-01 06:00:00 ... 2015-01-01 00:00:00
Data variables:
    p_1          (t_1, msl, latitude_1, longitude_1) float32 6GB ...
Attributes:
    history:  Sun Jul 27 16:49:11 BST 2025 - CONVSH V1.94 03-May-2018


In [23]:
ds_selected = ds['p_1']
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/p_1.zarr', mode='w')

### Future

In [61]:
ds = xr.open_dataset(out+'SSP585/apc/p_1.zarr')
print(ds)

<xarray.Dataset> Size: 14GB
Dimensions:      (latitude_1: 144, longitude_1: 192, msl: 1, t_1: 123840)
Coordinates:
  * latitude_1   (latitude_1) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * msl          (msl) float32 4B 0.0
  * t_1          (t_1) object 991kB 2015-01-01 06:00:00 ... 2101-01-01 00:00:00
Data variables:
    p_1          (t_1, msl, latitude_1, longitude_1) float32 14GB ...
Attributes:
    history:  Sun Sep  7 21:59:19 BST 2025 - CONVSH V1.94 03-May-2018


In [62]:
ds_selected = ds['p_1']
ds_selected.sel(t_1=slice(future_start, future_end))
rename_vars = {
    "t_1": "t"
}
ds_selected = ds_selected.rename(rename_vars)
ds_selected.to_zarr(out + 'final/p_1_future.zarr', mode='w')

### Select training and test data

In [4]:
### Choose variables here ###
train=False
train_years = (1985, 2014)
future_test_years = (2097, 2100)

In [5]:
load_from = '/hkfs/work/workspace/scratch/xo8179-ukesm/regridded/'
appendix = '' if train else '_future'

In [6]:
!ls /hkfs/work/workspace/scratch/xo8179-ukesm/regridded/

dz_dt_future.zarr	    q.zarr		      u_surface_future.zarr
dz_dt.zarr		    rvor_2.zarr		      u_surface.zarr
ht_1_future.zarr	    rvor_future.zarr	      u.zarr
ht_1.zarr		    temp_future.zarr	      v_future.zarr
p_1_future.zarr		    temp_surface_future.zarr  v_surface_future.zarr
p_1.zarr		    temp_surface.zarr	      v_surface.zarr
precip_surface_future.zarr  temp_t1.zarr	      v.zarr
precip_surface.zarr	    temp.zarr
q_future.zarr		    u_future.zarr


In [7]:
u = xr.open_dataset(load_from + f'u{appendix}.zarr')

v = xr.open_dataset(load_from + f'v{appendix}.zarr')

temp = xr.open_dataset(load_from + f'temp{appendix}.zarr')

q = xr.open_dataset(load_from + f'q{appendix}.zarr')

rvor = xr.open_dataset(load_from + f'rvor{appendix}.zarr')

geopot_500 = xr.open_dataset(load_from + f'ht_1{appendix}.zarr')

slp = xr.open_dataset(load_from + f'p_1{appendix}.zarr')

dz_dt = xr.open_dataset(load_from + f'dz_dt{appendix}.zarr')

u_surf = xr.open_dataset(load_from + f'u_surface{appendix}.zarr')

v_surf = xr.open_dataset(load_from + f'v_surface{appendix}.zarr')

temp_surf = xr.open_dataset(load_from + f'temp_surface{appendix}.zarr')

precip_surf = xr.open_dataset(load_from + f'precip_surface{appendix}.zarr')

In [25]:
print(u)

<xarray.Dataset> Size: 2GB
Dimensions:    (latitude: 64, level: 6, longitude: 128, time: 8641)
Coordinates:
  * latitude   (latitude) float64 512B -90.0 -87.14 -84.29 ... 84.29 87.14 90.0
  * level      (level) float32 24B 200.0 300.0 500.0 700.0 850.0 925.0
  * longitude  (longitude) float64 1kB 0.0 2.812 5.625 ... 351.6 354.4 357.2
  * time       (time) object 69kB 2095-01-01 00:00:00 ... 2101-01-01 00:00:00
Data variables:
    u          (time, level, longitude, latitude) float32 2GB ...


In [26]:
dsera5 = xr.open_dataset('/hkfs/work/workspace/scratch/xo8179-neural_lam/data/data/global_era5_1980_2022_6h-128x64_equiangular_with_poles_conservative/fields.zarr')

In [27]:
print(dsera5)

<xarray.Dataset> Size: 167GB
Dimensions:                  (time: 61400, longitude: 128, latitude: 64,
                              level: 13)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level                    (level) int64 104B 50 100 150 200 ... 850 925 1000
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) datetime64[ns] 491kB 1979-12-23 ... 2021-...
Data variables: (12/13)
    10m_u_component_of_wind  (time, longitude, latitude) float32 2GB ...
    10m_v_component_of_wind  (time, longitude, latitude) float32 2GB ...
    2m_temperature           (time, longitude, latitude) float32 2GB ...
    geopotential             (time, level, longitude, latitude) float32 26GB ...
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...
    ...                       ...
    specific_humidity        

In [16]:
!ls /hkfs/work/workspace/scratch/xo8179-ukesm/global_2097_2100_equiangular_wp_conservative

In [8]:
import cftime
train_dir = f'/hkfs/work/workspace/scratch/xo8179-ukesm/global_{train_years[0]}_{train_years[1]}_equiangular_wp_conservative/fields.zarr'
future_test_dir = f'/hkfs/work/workspace/scratch/xo8179-ukesm/global_{future_test_years[0]}_{future_test_years[1]}_equiangular_wp_conservative/fields.zarr'
if train:
    slice_start = cftime.Datetime360Day(train_years[0], 1, 1, 6)
    slice_end = cftime.Datetime360Day(train_years[1], 12, 30, 18)
    current_dir = train_dir
else:
    slice_start = cftime.Datetime360Day(future_test_years[0], 1, 1, 0)
    slice_end = cftime.Datetime360Day(future_test_years[1], 12, 30, 18)
    current_dir = future_test_dir
print(f'Saving {slice_start} until {slice_end}\nto Directory: {current_dir}')

Saving 2097-01-01 00:00:00 until 2100-12-30 18:00:00
to Directory: /hkfs/work/workspace/scratch/xo8179-ukesm/global_2097_2100_equiangular_wp_conservative/fields.zarr


In [23]:
temp_subset = temp.sel(time=slice(slice_start, slice_end))
rename_vars = { "temp": "temperature" }
temp_subset = temp_subset.rename(rename_vars)
temp_subset.to_zarr(current_dir, mode='w')

In [29]:
u_subset = u.sel(time=slice(slice_start, slice_end))
print(u_subset)
rename_vars = { "u": "u_component_of_wind" }
u_subset = u_subset.rename(rename_vars)
u_subset.to_zarr(current_dir, mode='a')

<xarray.Dataset> Size: 1GB
Dimensions:    (latitude: 64, level: 6, longitude: 128, time: 5760)
Coordinates:
  * latitude   (latitude) float64 512B -90.0 -87.14 -84.29 ... 84.29 87.14 90.0
  * level      (level) float32 24B 200.0 300.0 500.0 700.0 850.0 925.0
  * longitude  (longitude) float64 1kB 0.0 2.812 5.625 ... 351.6 354.4 357.2
  * time       (time) object 46kB 2097-01-01 00:00:00 ... 2100-12-30 18:00:00
Data variables:
    u          (time, level, longitude, latitude) float32 1GB ...


In [30]:
v_subset = v.sel(time=slice(slice_start, slice_end))
rename_vars = { "v": "v_component_of_wind" }
v_subset = v_subset.rename(rename_vars)
v_subset.to_zarr(current_dir, mode='a')

In [31]:
q_subset = q.sel(time=slice(slice_start, slice_end))
rename_vars = { "q": "specific_humidity" }
q_subset = q_subset.rename(rename_vars)
q_subset.to_zarr(current_dir, mode='a')

In [32]:
dz_dt_subset = dz_dt.sel(time=slice(slice_start, slice_end))
rename_vars = { "dz_dt": "vertical_velocity" }
dz_dt_subset = dz_dt_subset.rename(rename_vars)
dz_dt_subset.to_zarr(current_dir, mode='a')

In [33]:
rvor_subset = rvor.sel(time=slice(slice_start, slice_end))
rename_vars = { "rvor_2": "relative_vorticity" }
rvor_subset = rvor_subset.rename(rename_vars)
rvor_subset.to_zarr(current_dir, mode='a')

In [43]:
print(rvor_subset)
print(u_surf_subset)

<xarray.Dataset> Size: 8GB
Dimensions:             (latitude: 64, level: 6, longitude: 128, time: 43199)
Coordinates:
  * latitude            (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * level               (level) float32 24B 200.0 300.0 500.0 700.0 850.0 925.0
  * longitude           (longitude) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * time                (time) object 346kB 1985-01-01 06:00:00 ... 2014-12-3...
Data variables:
    relative_vorticity  (time, level, longitude, latitude) float32 8GB ...
<xarray.Dataset> Size: 1GB
Dimensions:    (latitude: 64, longitude: 128, time: 43199)
Coordinates:
  * latitude   (latitude) float64 512B -90.0 -87.14 -84.29 ... 84.29 87.14 90.0
  * longitude  (longitude) float64 1kB 0.0 2.812 5.625 ... 351.6 354.4 357.2
  * time       (time) object 346kB 1985-01-01 06:00:00 ... 2014-12-30 18:00:00
Data variables:
    u          (time, longitude, latitude) float32 1GB ...


In [34]:
u_surf_subset = u_surf.sel(time=slice(slice_start, slice_end))
rename_vars = { "u": "surface_u_component_of_wind" }
u_surf_subset = u_surf_subset.rename(rename_vars)
u_surf_subset.to_zarr(current_dir, mode='a')

In [35]:
v_surf_subset = v_surf.sel(time=slice(slice_start, slice_end))
rename_vars = { "v": "surface_v_component_of_wind" }
v_surf_subset = v_surf_subset.rename(rename_vars)
v_surf_subset.to_zarr(current_dir, mode='a')

In [36]:
temp_surf_subset = temp_surf.sel(time=slice(slice_start, slice_end))
rename_vars = { "temp": "surface_temperature" }
temp_surf_subset = temp_surf_subset.rename(rename_vars)
temp_surf_subset.to_zarr(current_dir, mode='a')

In [9]:
precip_surf_subset = precip_surf.sel(time=slice(slice_start, slice_end))
rename_vars = { "precip_accum_6h": "total_precipitation_6hr" }
precip_surf_subset = precip_surf_subset.rename(rename_vars)
print(precip_surf_subset)
precip_surf_subset.to_zarr(current_dir, mode='a')

<xarray.Dataset> Size: 189MB
Dimensions:                  (latitude: 64, longitude: 128, time: 5760)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * time                     (time) object 46kB 2097-01-01 00:00:00 ... 2100-...
Data variables:
    total_precipitation_6hr  (time, longitude, latitude) float32 189MB ...


In [40]:
geopot_500_subset = geopot_500.sel(time=slice(slice_start, slice_end))
rename_vars = { "ht_1": "geopotential_500" }
geopot_500_subset = geopot_500_subset.rename(rename_vars)
print(geopot_500_subset)

<xarray.Dataset> Size: 189MB
Dimensions:           (time: 5760, p: 1, longitude: 128, latitude: 64)
Coordinates:
  * latitude          (latitude) float64 512B -90.0 -87.14 -84.29 ... 87.14 90.0
  * longitude         (longitude) float64 1kB 0.0 2.812 5.625 ... 354.4 357.2
  * p                 (p) float32 4B 500.0
  * time              (time) object 46kB 2097-01-01 00:00:00 ... 2100-12-30 1...
Data variables:
    geopotential_500  (time, p, longitude, latitude) float32 189MB ...


In [41]:
g2d = geopot_500_subset['geopotential_500'].squeeze('p', drop=True)
g2d.name = 'geopotential_500'  # rename cleanly

g2d.to_zarr(current_dir, mode='a')

In [42]:
slp_subset = slp.sel(time=slice(slice_start, slice_end))
rename_vars = { "p_1": "mean_sea_level_pressure" }
slp_subset = slp_subset.rename(rename_vars)
print(slp_subset)

<xarray.Dataset> Size: 189MB
Dimensions:                  (latitude: 64, longitude: 128, msl: 1, time: 5760)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
  * msl                      (msl) float32 4B 0.0
  * time                     (time) object 46kB 2097-01-01 00:00:00 ... 2100-...
Data variables:
    mean_sea_level_pressure  (time, msl, longitude, latitude) float32 189MB ...


In [43]:
g2d = slp_subset['mean_sea_level_pressure'].squeeze('msl', drop=True)
g2d.name = 'mean_sea_level_pressure'  # rename cleanly

g2d.to_zarr(current_dir, mode='a')

In [44]:
dsera5_subset = dsera5[['geopotential_at_surface', 'land_sea_mask']]

In [45]:
print(dsera5_subset)

<xarray.Dataset> Size: 67kB
Dimensions:                  (longitude: 128, latitude: 64)
Coordinates:
  * latitude                 (latitude) float64 512B -90.0 -87.14 ... 87.14 90.0
  * longitude                (longitude) float64 1kB 0.0 2.812 ... 354.4 357.2
Data variables:
    geopotential_at_surface  (longitude, latitude) float32 33kB ...
    land_sea_mask            (longitude, latitude) float32 33kB ...


In [46]:
dsera5_subset.to_zarr(current_dir, mode='a')

In [10]:
final = xr.open_dataset(current_dir)
print(final)

<xarray.Dataset> Size: 8GB
Dimensions:                      (time: 5760, longitude: 128, latitude: 64,
                                  level: 6)
Coordinates:
  * latitude                     (latitude) float64 512B -90.0 -87.14 ... 90.0
  * level                        (level) float32 24B 200.0 300.0 ... 850.0 925.0
  * longitude                    (longitude) float64 1kB 0.0 2.812 ... 357.2
  * time                         (time) object 46kB 2097-01-01 00:00:00 ... 2...
Data variables: (12/14)
    geopotential_500             (time, longitude, latitude) float32 189MB ...
    geopotential_at_surface      (longitude, latitude) float32 33kB ...
    land_sea_mask                (longitude, latitude) float32 33kB ...
    mean_sea_level_pressure      (time, longitude, latitude) float32 189MB ...
    relative_vorticity           (time, level, longitude, latitude) float32 1GB ...
    specific_humidity            (time, level, longitude, latitude) float32 1GB ...
    ...                     

# Some test on times

In [8]:

import numpy as np
import pandas as pd
fields_group = final

# Lat-lon
grid_lat_vals = np.array(
    fields_group["latitude"], dtype=np.float32
)  # (num_lat,)
grid_lon_vals = np.array(
    fields_group["longitude"], dtype=np.float32
)  # (num_long,)
num_lat = grid_lat_vals.shape[0]
num_lon = grid_lon_vals.shape[0]

# Construct timestamps
# Time 0 here is 1959-01-01, 00:00
print("Constructing timestamps")
time_vals = fields_group.coords["time"].values

# Handle CFTime vs datetime64 calendars
if np.issubdtype(time_vals.dtype, np.datetime64):
    # Regular datetime64 (ERA5-style)
    timestamps = time_vals.astype("datetime64[s]")
    seconds_since_epoch = timestamps.astype("int64")
    timestamps_for_graphcast = time_vals.astype("datetime64[ns]")
else:
    # CFTime (360-day or other non-Gregorian calendars)
    timestamps = time_vals  # keep as cftime.Datetime360Day objects
    
    # CFTime → approximate datetime64 (drop calendar info)
    dt_seconds = (time_vals[1] - time_vals[0]).total_seconds()
    freq_hours = int(round(dt_seconds / 3600))
    
    # Create a continuous datetime64 series with same number of steps
    timestamps_for_graphcast = pd.date_range(
        start="1990-01-01 00:00:00",
        periods=len(time_vals),
        freq=f"{freq_hours}H"
    )

    # Convert to numeric seconds manually
    epoch = cftime.Datetime360Day(1959, 1, 1, 0, 0, 0)
    seconds_since_epoch = np.array([
        (t - epoch).total_seconds() for t in timestamps
    ], dtype=np.float64)


Constructing timestamps


/scratch/slurm_tmpdir/job_3568183/ipykernel_4062958/3261686653.py:35: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  timestamps_for_graphcast = pd.date_range(


In [10]:
print(timestamps_for_graphcast[-1])

2019-07-27 00:00:00


# ap8 supplementary

In [5]:
ap8_supplementary_folder = "/hkfs/work/workspace/scratch/ou4895-for_lukas/Hist/ap8_supplement"

In [6]:
ds = load_x_files_from_folder(ap8_supplementary_folder, 0, 1)

Found 1258 in specified folder.
Attempting to load 1 files...


In [7]:
print(ds)

<xarray.Dataset> Size: 434MB
Dimensions:      (longitude: 192, latitude: 144, surface: 1, t: 80,
                  unspecified: 1, t_1: 80, ht: 1, pseudo: 5, pseudo_1: 3,
                  pseudo_2: 5, pseudo_3: 14, level6: 4)
Coordinates:
  * longitude    (longitude) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * surface      (surface) float32 4B 0.0
  * t            (t) object 640B 1979-01-01 01:30:00 ... 1979-01-10 22:30:00
  * unspecified  (unspecified) float32 4B -1.0
  * t_1          (t_1) object 640B 1979-01-01 03:00:00 ... 1979-01-11 00:00:00
  * ht           (ht) float32 4B -1.0
  * pseudo       (pseudo) float32 20B 101.0 102.0 103.0 201.0 202.0
  * pseudo_1     (pseudo_1) float32 12B 3.0 301.0 302.0
  * pseudo_2     (pseudo_2) float32 20B 4.0 401.0 402.0 501.0 502.0
  * pseudo_3     (pseudo_3) float32 56B 6.0 7.0 8.0 9.0 ... 908.0 909.0 910.0
  * level6       (level6) float32 16B 0.05 0.225 0.6

In [10]:
ds = load_x_files_from_folder(ap8_supplementary_folder, 48, 49)
print(ds)

Found 1258 in specified folder.
Attempting to load 1 files...
<xarray.Dataset> Size: 434MB
Dimensions:      (longitude: 192, latitude: 144, surface: 1, t: 80,
                  unspecified: 1, t_1: 80, ht: 1, pseudo: 5, pseudo_1: 3,
                  pseudo_2: 5, pseudo_3: 14, level6: 4)
Coordinates:
  * longitude    (longitude) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * surface      (surface) float32 4B 0.0
  * t            (t) object 640B 1980-05-01 01:30:00 ... 1980-05-10 22:30:00
  * unspecified  (unspecified) float32 4B -1.0
  * t_1          (t_1) object 640B 1980-05-01 03:00:00 ... 1980-05-11 00:00:00
  * ht           (ht) float32 4B -1.0
  * pseudo       (pseudo) float32 20B 101.0 102.0 103.0 201.0 202.0
  * pseudo_1     (pseudo_1) float32 12B 3.0 301.0 302.0
  * pseudo_2     (pseudo_2) float32 20B 4.0 401.0 402.0 501.0 502.0
  * pseudo_3     (pseudo_3) float32 56B 6.0 7.0 8.0 9.0 ... 908.0 90

In [30]:
selected = ds['temp'].sel(latitude=-89.375, longitude=0.9375)

In [31]:
print(selected)

<xarray.DataArray 'temp_5' (t_1: 80, surface: 1)> Size: 320B
dask.array<getitem, shape=(80, 1), dtype=float32, chunksize=(80, 1), chunktype=numpy.ndarray>
Coordinates:
    longitude  float32 4B 0.9375
    latitude   float32 4B -89.38
  * surface    (surface) float32 4B 0.0
  * t_1        (t_1) object 640B 1980-05-01 03:00:00 ... 1980-05-11 00:00:00
Attributes:
    source:         Unified Model Output (Vn10.9):
    name:           temp_5
    title:          Temperature T
    date:           01/01/50
    time:           00:00
    long_name:      Temperature T
    standard_name:  surface_temperature
    units:          K
    valid_min:      182.78882
    valid_max:      355.08008


In [35]:
print(selected.sel(t_1='1980-05-01 12:00:00').values)

[[215.50952]]


In [16]:
ds = load_x_files_from_folder(apk_folder_future, -20, -19)

Found 1019 in specified folder.
Attempting to load 1 files...


In [17]:
print(ds)

<xarray.Dataset> Size: 106MB
Dimensions:      (longitude: 192, latitude: 144, ht: 1, t: 240,
                  longitude_1: 192, latitude_1: 145, surface: 1, hybrid_ht: 1)
Coordinates:
  * longitude    (longitude) float32 768B 0.0 1.875 3.75 ... 354.4 356.2 358.1
  * latitude     (latitude) float32 576B -89.38 -88.12 -86.88 ... 88.12 89.38
  * ht           (ht) float32 4B -1.0
  * t            (t) object 2kB 2099-01-01 01:30:00 ... 2099-01-30 22:30:00
  * longitude_1  (longitude_1) float32 768B 0.9375 2.812 4.688 ... 357.2 359.1
  * latitude_1   (latitude_1) float32 580B -90.0 -88.75 -87.5 ... 88.75 90.0
  * surface      (surface) float32 4B 0.0
  * hybrid_ht    (hybrid_ht) float32 4B 20.0
Data variables:
    u            (t, ht, latitude, longitude) float32 27MB dask.array<chunksize=(240, 1, 144, 192), meta=np.ndarray>
    v            (t, ht, latitude_1, longitude_1) float32 27MB dask.array<chunksize=(240, 1, 145, 192), meta=np.ndarray>
    precip       (t, surface, latitude, longitu

In [23]:
# Open dataset (with dask)
ds = xr.open_dataset("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_long_chunks/fields.zarr")



In [31]:
from dask.diagnostics import ProgressBar
out = "/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_40_chunks/fields.zarr"
chunks = 40

In [34]:
# Rechunk time dimension to 1
ds_select = ds_ukesm["specific_humidity"]
ds_rechunked = ds_select.chunk({"time": chunks, "longitude": 128, "latitude": 64, 'level':6})
ds_rechunked.encoding.clear()

# Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
with ProgressBar():
    ds_rechunked.to_zarr(out, mode="w")

print("✅ Rechunked and saved with time chunks = 1")

[########################################] | 100% Completed | 6.31 ss
✅ Rechunked and saved with time chunks = 1


In [35]:
# Rechunk time dimension to 1
ds_select = ds[["mean_sea_level_pressure",
    "geopotential_500",
    "total_precipitation_6hr",
    "surface_temperature",
    "surface_u_component_of_wind",
    "surface_v_component_of_wind"]]
from dask.diagnostics import ProgressBar
with ProgressBar():
    ds_rechunked = ds_select.chunk({"time": chunks, "longitude": 128, "latitude": 64})
    for v in ds_rechunked.variables:
        ds_rechunked[v].encoding.clear()
    # Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
    ds_rechunked.to_zarr(out, mode="a")
    
    print("✅ Rechunked and saved with time chunks = 1")

[########################################] | 100% Completed | 55.88 s
✅ Rechunked and saved with time chunks = 1


In [36]:
# Rechunk time dimension to 1
ds_select = ds["temperature"]
ds_rechunked = ds_select.chunk({"time": chunks, "longitude": 128, "latitude": 64, 'level':6})
ds_rechunked.encoding.clear()
# Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
with ProgressBar():
    ds_rechunked.to_zarr(out, mode="a")

print("✅ Rechunked and saved with time chunks = 1")

[########################################] | 100% Completed | 8.19 ss
✅ Rechunked and saved with time chunks = 1


# Continue here

In [37]:
for var in ["u_component_of_wind",
    "v_component_of_wind",
    "vertical_velocity",
    "relative_vorticity"]:
    # Rechunk time dimension to 1
    ds_select = ds[var]
    ds_rechunked = ds_select.chunk({"time": chunks,"longitude": 128, "latitude": 64, 'level':6})
    ds_rechunked.encoding.clear()
    # Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
    with ProgressBar():
        ds_rechunked.to_zarr(out, mode="a")
    print(f"✅ Rechunked and saved with time chunks = 1: {var}")

[########################################] | 100% Completed | 7.47 ss
✅ Rechunked and saved with time chunks = 1: u_component_of_wind
[########################################] | 100% Completed | 8.70 ss
✅ Rechunked and saved with time chunks = 1: v_component_of_wind
[########################################] | 100% Completed | 75.32 s
✅ Rechunked and saved with time chunks = 1: vertical_velocity
[########################################] | 100% Completed | 6.31 ss
✅ Rechunked and saved with time chunks = 1: relative_vorticity


In [38]:
# Rechunk time dimension to 1
ds_select = ds[["geopotential_at_surface",
	"land_sea_mask"]]
ds_rechunked = ds_select.chunk({"longitude": 128, "latitude": 64})
for v in ds_rechunked.variables:
    ds_rechunked[v].encoding.clear()
# Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
ds_rechunked.to_zarr(out, mode="a")

print("✅ Rechunked and saved with time chunks = 1")

✅ Rechunked and saved with time chunks = 1


In [66]:
# Open dataset (with dask)
ds = xr.open_zarr("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_long_chunks/forcing.zarr")

ds_rechunked = ds.chunk({"time": chunks})
for v in ds_rechunked.variables:
    ds_rechunked[v].encoding.clear()
with ProgressBar():
        ds_rechunked.to_zarr("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_40_chunks/forcing.zarr", mode="w")

[########################################] | 100% Completed | 66.29 s


In [67]:
!cp -r /hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_long_chunks/static /hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_40_chunks/static

In [8]:
import shutil

shutil.rmtree("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative/forcing.zarr", ignore_errors=True)

In [58]:
ds_ukesm = xr.open_zarr('/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative/fields.zarr')
#ds_ukesm = xr.open_zarr(out)
ds_nextgems = xr.open_zarr('/hkfs/work/workspace/scratch/xo8179-neural_lam/data/data/global_nextgems_1990_2020_6h-128x64_equiangular_with_poles_conservative/fields.zarr')

In [ ]:

print(ds_ukesm.chunks)

In [ ]:
print(ds_nextgems.chunks)

In [50]:
print(ds_ukesm.time.encoding)
print(len(ds_ukesm.time))

{'chunks': (44010,), 'preferred_chunks': {'time': 44010}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'units': 'days since 1850-01-01', 'calendar': '360_day', 'dtype': dtype('float32')}
43199


In [43]:
print(ds_nextgems.time.encoding)
print(len(ds_nextgems.time))

{'chunks': (14608,), 'preferred_chunks': {'time': 14608}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, 'units': 'seconds since 1970-01-01', 'calendar': 'proleptic_gregorian', 'dtype': dtype('int64')}
43828


In [ ]:
# target_chunks = {
#     "specific_humidity": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "temperature": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "u_component_of_wind": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "v_component_of_wind": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "vertical_velocity": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "relative_vorticity": {"time": 1, "longitude": 128, "latitude": 64, "level": 6},
#     "mean_sea_level_pressure": {"time": 1, "longitude": 128, "latitude": 64},
#     "geopotential_500": {"time": 1, "longitude": 128, "latitude": 64},
#     "total_precipitation_6hr": {"time": 1, "longitude": 128, "latitude": 64},
#     "surface_temperature": {"time": 1, "longitude": 128, "latitude": 64},
#     "surface_u_component_of_wind": {"time": 1, "longitude": 128, "latitude": 64},
#     "surface_v_component_of_wind": {"time": 1, "longitude": 128, "latitude": 64},
#     "geopotential_at_surface": {"longitude": 128, "latitude": 64},
# 	"land_sea_mask": {"longitude": 128, "latitude": 64},
#     "time": None,  # don't rechunk this array
#     "longitude": None,
#     "latitude": None,
#     "level": None,
# }
# import xarray as xr
# import rechunker

# source = xr.open_zarr("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_long_chunks/fields.zarr")
# target_store = "/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative/fields_new.zarr"
# temp_store = "/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative/temp.zarr"

# import shutil

# shutil.rmtree(target_store, ignore_errors=True)
# shutil.rmtree(temp_store, ignore_errors=True)


# # Plan rechunking
# plan = rechunker.rechunk(
#     source, 
#     target_chunks=target_chunks, 
#     target_store=target_store,
#     temp_store=temp_store,
#     max_mem="200GB"
# )

# # Execute
# from dask.diagnostics import ProgressBar

# with ProgressBar():
#     plan.execute()

[###########                             ] | 29% Completed | 90.24 sms

In [12]:
# Open dataset (with dask)
ds = xr.open_dataset("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative_long_chunks/forcing.zarr")
print(ds)

<xarray.Dataset> Size: 7GB
Dimensions:                        (time: 43199, longitude: 128, latitude: 64,
                                    forcing_var: 5)
Coordinates:
  * forcing_var                    (forcing_var) object 40B 'toa_incident_rad...
  * latitude                       (latitude) float64 512B -90.0 -87.14 ... 90.0
  * longitude                      (longitude) float64 1kB 0.0 2.812 ... 357.2
  * time                           (time) object 346kB 1985-01-01 06:00:00 .....
Data variables:
    __xarray_dataarray_variable__  (time, longitude, latitude, forcing_var) float32 7GB ...


In [13]:
print(ds['time'].encoding)

{'chunks': (44010,), 'preferred_chunks': {'time': 44010}, 'compressor': Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0), 'filters': None, '_FillValue': nan, 'units': 'days since 1850-01-01', 'calendar': '360_day', 'dtype': dtype('float32')}


In [ ]:

# Rechunk time dimension to 1
ds_rechunked = ds.chunk({"time": 1})

# Option 2 (RECOMMENDED): Save as Zarr (more scalable for large data)
ds_rechunked.to_zarr("/hkfs/work/workspace/scratch/xo8179-ukesm/global_1985_2014_equiangular_wp_conservative/forcing.zarr", mode="w")

print("✅ Rechunked and saved with time chunks = 1")